In [11]:
import numpy as np
import pandas as pd
import openpyxl
import json, math, os, re, time, unicodedata

# Part B — Extensive margin: mode choice

### 1. Cross-tabulate main_mode by site and year. Report the gross flows in both directions (the amount of people switching from car to another mode, and the opposite) and the net mode share percentage change.

#### Required output: Table B1 with mode shares by site and year and the numbers switching car-to-non-car and non-car-to-car

Nettoyage de Nina : (sans les com)

In [12]:
raw = pd.read_csv("parking_data.csv")

miss = raw[raw["gross_income_chf"].isna()]
n_years = miss.groupby("employee_id").size()

for col in ["site", "home_commune", "home_postcode", "distance_km", "age",
            "n_children", "fte", "gross_income_chf"]:
    n = (raw.groupby("employee_id")[col].nunique() > 1).sum()
    print(f"{col:<18} change pour {n} employés")

def normalize(name):
    """Majuscules, sans accents ni ponctuation : 'Chavannes-près-Renens' -> 'CHAVANNES PRES RENENS'."""
    name = unicodedata.normalize("NFKD", str(name).upper().strip())
    name = "".join(ch for ch in name if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]", " ", name)).strip()

check = raw.assign(commune_clean=raw["home_commune"].apply(normalize))
print("Orthographes différentes :", raw["home_commune"].nunique(), "-> communes après harmonisation :",
      check["commune_clean"].nunique())
pd.set_option("display.max_rows", 100)
check.groupby("commune_clean").agg(
    site=("site", "unique"),
    variantes=("home_commune", lambda s: sorted(s.unique())),
    npa=("home_postcode", "unique"),
    employes=("employee_id", "nunique"),
)

raw["commune_clean"] = raw["home_commune"].apply(normalize)
raw["commune_clean_raw"] = raw["commune_clean"]
raw["home_postcode_raw"] = raw["home_postcode"]

raw["commune_clean"] = raw["commune_clean"].replace({"SAVIEZE": "SAVIESE"})   # faute d'orthographe
raw.loc[raw["commune_clean"] == "SALINS", "home_postcode"] = 1991      

raw["income_imputed"] = raw["gross_income_chf"].isna()
raw["gross_income_chf"] = raw.groupby("employee_id")["gross_income_chf"].transform(lambda s: s.fillna(s.mean()))
raw["income_imputed"] &= raw["gross_income_chf"].notna()
raw["car_days_raw"] = raw["car_days_per_week"]
raw["car_days_per_week"] = raw["car_days_per_week"].clip(upper=5)
communes = (raw.groupby(["commune_clean", "site"])
               .agg(n_employees=("employee_id", "nunique"),
                    postcode=("home_postcode", "first"),
                    mean_distance_km=("distance_km", "mean"))
               .reset_index())
assert len(communes) == 39 and communes.n_employees.sum() == 1000
communes

#raw.to_excel("donnes_clean.xlsx", index=False)

site               change pour 0 employés
home_commune       change pour 45 employés
home_postcode      change pour 0 employés
distance_km        change pour 0 employés
age                change pour 0 employés
n_children         change pour 0 employés
fte                change pour 0 employés
gross_income_chf   change pour 0 employés
Orthographes différentes : 62 -> communes après harmonisation : 39


,commune_clean,site,n_employees,postcode,mean_distance_km
0,ARBAZ,Sion,6,1974,11.483333
1,ARDON,Sion,11,1957,13.809091
2,AYENT,Sion,15,1966,12.773333
3,BOTTENS,Lausanne,15,1041,14.440000
4,BRAMOIS,Sion,16,1967,5.318750
5,BUSSIGNY,Lausanne,43,1030,9.788372
6,CHAVANNES PRES RENENS,Lausanne,20,1022,7.110000
7,CHESEAUX SUR LAUSANNE,Lausanne,21,1033,8.547619
8,CONTHEY,Sion,20,1964,7.225000
9,CORCELLES LE JORAT,Lausanne,13,1082,19.423077


In [10]:
print(raw.columns)

Index(['employee_id', 'site', 'year', 'parking_fee_chf', 'main_mode',
       'uses_car', 'car_days_per_week', 'home_commune', 'home_postcode',
       'distance_km', 'age', 'gender', 'n_children', 'fte', 'gross_income_chf',
       'commune_clean', 'commune_clean_raw', 'home_postcode_raw',
       'income_imputed', 'car_days_raw'],
      dtype='object')


In [ ]:
cross_table = pd.crosstab([raw['site'], raw['year']], raw['main_mode'])
#In percentage : 
shares = pd.crosstab([raw['site'], raw['year']], raw['main_mode'], normalize='index') * 100

#Changements nets : 
net_change_nb = cross_table.xs(2025, level='year') - cross_table.xs(2024, level='year')   # nb final - nb début

net_change_pp = shares.xs(2025, level='year') - shares.xs(2024, level='year')      # en points de %

# Changements bruts : 
wide = raw.pivot(index='employee_id', columns='year', values='main_mode') # pour voir le changement chez chaque employé
wide['site'] = raw.drop_duplicates('employee_id').set_index('employee_id')['site']

transitions = {s: pd.crosstab(g[2024], g[2025], margins=True) for s, g in wide.groupby('site')}

flows = wide.groupby('site')[[2024, 2025]].apply(lambda g: pd.Series({
    'car_to_noncar': ((g[2024] == 'car') & (g[2025] != 'car')).sum(),
    'noncar_to_car': ((g[2024] != 'car') & (g[2025] == 'car')).sum(),
})) # pour savoir qui a changé 

flows['net_car'] = flows['noncar_to_car'] - flows['car_to_noncar']

# --- Table B1 ---
table_B1 = shares.round(1).unstack('year')
table_B1[('car', 'Δ pp')] = net_change_pp['car'].round(1)
flows.columns = pd.MultiIndex.from_product([['flows'], flows.columns]) 
table_B1 = table_B1.join(flows)
table_B1


bike         bus         car                     flows  \
          2024  2025  2024  2025  2024  2025  Δ pp car_to_noncar   
site                                                               
Lausanne  12.1  14.8  34.0  41.4  53.9  43.9 -10.0            80   
Sion      14.5  14.5  27.0  24.0  58.5  61.5   3.0             0   

                                
         noncar_to_car net_car  
site                            
Lausanne             0     -80  
Sion                 6       6

### 2. Compute the change in the car share in both percentage points and as an elasticity. Explain why these answer different questions, and identify which measure belongs in the policy memo.

In [27]:
# Part de la voiture (%) par site et année
car_share = shares['car'].unstack('year')
price = {'Lausanne': (2, 4), 'Sion': (2, 2)}

# Élasticité extensive en log : Δln(part voiture) / Δln(prix)
# -> même définition que l'équation (1) (log fee) et nécessaire pour que
#    ε_total = ε_extensive + ε_intensive soit exact en Partie C.4
def ext_elasticity(share_2024, share_2025, p0=2, p1=4):
    return np.log(share_2025 / share_2024) / np.log(p1 / p0)

# 1) Changement en points de pourcentage (pp)
car_share['delta_pp'] = car_share[2025] - car_share[2024]

# 2) Élasticité (seulement où le prix a changé : Lausanne)
car_share['delta_ln_price'] = [np.log(price[s][1] / price[s][0]) for s in car_share.index]
car_share['elasticity'] = [ext_elasticity(r[2024], r[2025], *price[s]) if r['delta_ln_price'] > 0 else np.nan
                           for s, r in car_share.iterrows()]   # Sion : prix inchangé -> pas d'élasticité

print(car_share.round(3))

year        2024    2025  delta_pp  delta_ln_price  elasticity
site                                                          
Lausanne  53.875  43.875     -10.0           0.693      -0.296
Sion      58.500  61.500       3.0           0.000         NaN


**Résultats**

| | Tarif | Part voiture 2024 | 2025 | Δ (pp) | Élasticité extensive |
|---|---|---|---|---|---|
| Lausanne | 2 → 4 CHF | 53,9 % | 43,9 % | **−10,0** | **−0,30** |
| Sion | 2 → 2 CHF | 58,5 % | 61,5 % | +3,0 | — (prix inchangé) |

Élasticité = Δln(part voiture) / Δln(prix) = ln(43,9 / 53,9) / ln(4 / 2) = −0,30. On prend la forme en log parce que c'est celle de l'équation (1) (log fee), et parce que la décomposition ε_total = ε_extensive + ε_intensive de la Partie C.4 n'est exacte qu'avec cette définition.

**Pourquoi ces deux mesures répondent à des questions différentes**
- **Points de pourcentage** : *« combien d'employés ont quitté la voiture avec cette mesure précise (2 → 4 CHF) ? »* À Lausanne, la part de la voiture baisse de 10 points, soit 80 employés sur 800. C'est un effet concret, qu'on peut traduire en places de parking libérées.
- **Élasticité** : *« à quel point la demande réagit-elle au prix, indépendamment de l'unité ? »* Une hausse de 10 % du tarif réduit la part de la voiture d'environ 3 %. La demande est **inélastique**. Cette mesure sert à comparer avec d'autres études et à extrapoler vers d'autres tarifs (Partie D). Attention, elle repose sur une seule variation de prix, très grande (+100 %).

**Dans le policy memo : les points de pourcentage.** Le memo répond à la question « combien de personnes arrêtent de conduire quand le prix double ? », et l'employeur raisonne en employés et en places de parking, pas en élasticités. L'élasticité reste dans le rapport technique, où elle sert aux scénarios 2026.

### 3. Describe heterogeneity in these results: disaggregate the previous result by transit-accessibility group, income tercile and number of children. Define how groups are formed and report group sample sizes.

#### Required output: Table B2 with, for every requested group, 2024 and 2025 car shares, the percentage-point change, extensive elasticity, and number of employees.

**Formation des groupes**
- **Revenu** : terciles = 3 groupes avec le même nombre d'employés, calculés sur les 1000 employés (une valeur par employé, et nous considérons que le revenu est constant entre les années). Les 4 employés sans revenu ne sont dans aucun tercile.
- **Enfants** : la variable a déjà 3 modalités (0, 1, 2 = « 2 ou plus »), gardées telles quelles.
- **Accessibilité TP** : `transit_group` de `transit_access.csv` (Partie A), c'est à dire des terciles de l'indice TP pondérés par les employés et calculés dans chaque site.

In [ ]:
# --- 1. Tercile de revenu ---
# Terciles = 3 groupes avec le même nombre d'employés
income_emp = raw.drop_duplicates('employee_id').set_index('employee_id')['gross_income_chf']

income_tercile, bins = pd.qcut(income_emp, 3, labels=['1_low', '2_medium', '3_high'], retbins=True)

raw['income_tercile'] = raw['employee_id'].map(income_tercile) #NaN pour les employés sans revenus.
print("Seuils des terciles (CHF) :", bins.round(0))

# --- 2. Nombre d'enfants : déjà 3 groupes (0, 1, 2 = "2 ou plus")
raw['children_group'] = raw['n_children'].map({0: '0', 1: '1', 2: '2+'})

# --- 3. Groupe d'accessibilité TP (Partie A) ---
transit = pd.read_csv('transit_access.csv', encoding='utf-8-sig')

raw = raw.drop(columns=['transit_index', 'transit_group'], errors='ignore')
raw = raw.merge(transit[['commune_clean', 'site', 'transit_index', 'transit_group']],
                on=['commune_clean', 'site'], how='left')
assert raw['transit_group'].notna().all() 

# --- Taille des groupes (nb d'employés) ---
emp = raw.drop_duplicates('employee_id')

for col in ['transit_group', 'income_tercile', 'children_group']:
    print('\n', pd.crosstab(emp[col], emp['site'], margins=True))
print("\nSans revenu (aucun tercile) :", emp['income_tercile'].isna().sum(), "employés")

**Taille des groupes (nombre d'employés)**

| Variable | Groupe | Lausanne | Sion | Total |
|---|---|---:|---:|---:|
| **Accessibilité TP** | 1_low | 285 | 66 | 351 |
| | 2_medium | 273 | 60 | 333 |
| | 3_high | 242 | 74 | 316 |
| | *Total* | *800* | *200* | *1000* |
| **Tercile de revenu** | 1_low (< 77 500 CHF) | 253 | 80 | 333 |
| | 2_medium (77 500 – 103 200 CHF) | 261 | 71 | 332 |
| | 3_high (> 103 200 CHF) | 282 | 49 | 331 |
| | *Total* | *796* | *200* | *996* |
| **Nombre d'enfants** | 0 | 421 | 113 | 534 |
| | 1 | 155 | 29 | 184 |
| | 2+ | 224 | 58 | 282 |
| | *Total* | *800* | *200* | *1000* |

*4 employés de Lausanne n'ont de revenu déclaré pour aucune des deux années : ils ne figurent dans aucun tercile, d'où un total de 996.*

In [ ]:
# --- Table B2 : part voiture par groupe, Lausanne (seul site où le prix change -> seule élasticité possible) ---
def group_table(data, col):
    t = data.pivot_table(index=col, columns='year', values='uses_car', aggfunc='mean', observed=True) * 100
    t['delta_pp'] = t[2025] - t[2024]
    t['elasticity'] = ext_elasticity(t[2024], t[2025])
    t['n_employees'] = data.groupby(col, observed=True)['employee_id'].nunique()
    t.index = pd.MultiIndex.from_product([[col], t.index.astype(str)], names=['variable', 'group'])
    return t

laus = raw[raw['site'] == 'Lausanne']
table_B2 = pd.concat([group_table(laus.assign(all='all'), 'all')] +
                     [group_table(laus, c) for c in ['transit_group', 'income_tercile', 'children_group']])
table_B2.round(2)

**Table B2 : part de la voiture par groupe, site de Lausanne (tarif 2 → 4 CHF)**

| Variable | Groupe | Part voiture 2024 | Part voiture 2025 | Δ (pp) | Élasticité extensive | n employés |
|---|---|---:|---:|---:|---:|---:|
| **Total** | Lausanne | 53,9 % | 43,9 % | −10,0 | −0,30 | 800 |
| **Accessibilité TP** | 1_low | 74,7 % | 68,8 % | −6,0 | −0,12 | 285 |
| | 2_medium | 44,0 % | 32,2 % | −11,7 | −0,45 | 273 |
| | 3_high | 40,5 % | 27,7 % | −12,8 | −0,55 | 242 |
| **Tercile de revenu** | 1_low (< 77 500 CHF) | 49,4 % | 36,4 % | −13,0 | −0,44 | 253 |
| | 2_medium (77 500 – 103 200 CHF) | 51,7 % | 43,3 % | −8,4 | −0,26 | 261 |
| | 3_high (> 103 200 CHF) | 60,6 % | 51,8 % | −8,9 | −0,23 | 282 |
| **Nombre d'enfants** | 0 | 45,1 % | 34,0 % | −11,2 | −0,41 | 421 |
| | 1 | 64,5 % | 49,7 % | −14,8 | −0,38 | 155 |
| | 2+ | 63,0 % | 58,5 % | −4,5 | −0,11 | 224 |

*Élasticité extensive = Δln(part voiture) / ln(4/2). Seul Lausanne est présenté, car le tarif de Sion n'a pas changé et aucune élasticité n'y est calculable. Ce sont des différences descriptives avant/après, sans contrôles : les groupes se recoupent (par exemple, revenu et accessibilité TP peuvent être corrélés). 4 employés sans revenu ne figurent dans aucun tercile.*

### 4. Estimate equation (1) sequentially: (a) Lausanne-only, (b) + Sion, (c) + controls, (d) + interactions

#### Required output: Table B3, a single four-column regression table.

**Équation (1) : modèle de probabilité linéaire (LPM)**
On régresse `uses_car` (0/1) par MCO. Un coefficient se lit comme une variation de **probabilité** d'utiliser la voiture.

**Variable de politique : log(fee)**, comme dans l'équation (1) de la consigne. Le coefficient *a* donne la variation de probabilité pour une hausse de 1 unité de log(fee). Comme le doublement du prix vaut Δlog(fee) = ln 2 ≈ 0,693, l'effet d'un doublement en pp vaut *a* × ln 2. Cette forme permet aussi de prédire d'autres tarifs en Partie D : de 4 à 6 CHF, Δp = *a* × ln(6/4).

#### (a) Lausanne seule, avant/après

$$\text{uses\_car}_{it} = \alpha + a \cdot \log(\text{fee}_{t}) + \varepsilon_{it} \qquad (\text{Lausanne, 2024–2025})$$

- Avec Lausanne seule, log(fee) ne prend que deux valeurs (ln 2 en 2024, ln 4 en 2025). Il contient donc exactement la même information qu'un indicateur `post` (2025 = 1). Les deux régressions sont équivalentes : *a* = β_post / ln 2, avec le même t de Student et le même R².
- **Erreurs standard groupées par employé** : chaque employé apparaît deux fois (2024 et 2025), donc ses deux observations ne sont pas indépendantes.
- **Limite** : *a* attribue *tout* le changement 2024 → 2025 au prix. Si la part voiture avait baissé de toute façon (météo, nouvelle ligne de bus, tendance générale), *a* le compterait à tort comme un effet du prix. C'est ce que Sion permet de vérifier en (b).

In [ ]:
import statsmodels.formula.api as smf

raw['post'] = (raw['year'] == 2025).astype(int) # 1 si 2025 0 sinon
raw['log_fee'] = np.log(raw['parking_fee_chf']) # variable de politique de l'équation (1)

laus = raw[raw['site'] == 'Lausanne']
cluster = lambda d: dict(cov_type='cluster', cov_kwds={'groups': d['employee_id']})   # SE groupées par employé

# (a) Lausanne seule : uses_car = α + a·log(fee)
m_a = smf.ols('uses_car ~ log_fee', data=laus).fit(**cluster(laus))
print(m_a.summary().tables[1])

a = m_a.params['log_fee']
print(f"\nEffet d'un doublement du prix (2 -> 4 CHF) : a × ln2 = {a * np.log(2):.3f}  -> {a * np.log(2) * 100:.1f} pp")

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6388      0.023     27.502      0.000       0.593       0.684
log_fee       -0.1443      0.015     -9.419      0.000      -0.174      -0.114

Effet d'un doublement du prix (2 -> 4 CHF) : a × ln2 = -0.100  -> -10.0 pp
Avec post : β = -0.100   (= a × ln2)


#### (b) Ajout de Sion comme site de comparaison

$$\text{uses\_car}_{it} = \alpha + a \cdot \log(\text{fee}_{st}) + \gamma \cdot \text{Lausanne}_s + \delta \cdot \text{Post}_t + \varepsilon_{it} \qquad (\text{2 sites, 2024–2025})$$


- **`lausanne`** (1 = Lausanne, 0 = Sion) : capte la **différence permanente** entre les deux sites (Sion a toujours plus de voitures, 58,5 % contre 53,9 % en 2024).
- **`post`** (1 = 2025) : capte la **tendance commune** 2024 → 2025, celle qui aurait touché tout le monde sans hausse de prix. Elle est mesurée par Sion, où le prix n'a pas changé : +3 pp.

**Hypothèse clé (tendances parallèles)** : sans la hausse, la part voiture à Lausanne aurait évolué comme à Sion (+3 pp).

In [ ]:
raw['lausanne'] = (raw['site'] == 'Lausanne').astype(int) # 1 = Lausanne (traité), 0 = Sion (contrôle)

# (b) Les deux sites : uses_car = α + a·log(fee) + γ·lausanne + δ·post
m_b = smf.ols('uses_car ~ log_fee + lausanne + post', data=raw).fit(**cluster(raw))
print(m_b.summary().tables[1])

b = m_b.params['log_fee']
print(f"\nEffet d'un doublement du prix : a × ln2 = {b * np.log(2):.3f}  -> {b * np.log(2) * 100:.1f} pp")

print(f"Comparaison avec (a) : {a * np.log(2) * 100:.1f} pp  ->  {b * np.log(2) * 100:.1f} pp")

#### (c) Comparaison avec contrôles

$$\text{uses\_car}_{it} = \alpha + a \cdot \log(\text{fee}_{st}) + \gamma \cdot \text{Lausanne}_s + \delta \cdot \text{Post}_t + X_i'\theta + \varepsilon_{it}$$

**Contrôles $X_i$ retenus et justification** : chacun influence le fait de prendre la voiture, et aucun n'est affecté par la hausse du prix, puisqu'ils sont mesurés et constants sur les deux années (ce ne sont donc pas des *bad controls*).

| Contrôle | Codage | Pourquoi |
|---|---|---|
| `transit_index` | indice continu de la Partie A (+ = meilleur accès TP) | exigé par la consigne : sans alternative TP, on prend la voiture |
| `distance_km` | km, continu | plus le trajet est long, moins le vélo ou le bus sont réalistes |
| `income_10k` | revenu brut / 10 000 CHF | capacité à payer le parking et à posséder une voiture |
| `n_children` | catégories 0 / 1 / 2+ (référence : 0 enfant) | contraintes familiales (crèche, école) |
| `age`, `female`, `fte` | continu / 0-1 / 0,6–1,0 | caractéristiques de base. Un temps partiel signifie moins de trajets |

**Échantillon** : les 4 employés sans revenu (8 lignes) sont exclus, soit 1992 observations.

In [ ]:
# Contrôles
raw['income_10k'] = raw['gross_income_chf'] / 10000 # revenu en dizaines de milliers de CHF
raw['female'] = (raw['gender'] == 'F').astype(int)

controls = 'transit_index + distance_km + income_10k + C(n_children) + age + female + fte'

est = raw.dropna(subset=['income_10k']) # sans les 4 employés sans revenu
print("Observations :", len(est), "| employés :", est['employee_id'].nunique())

# (c) Les deux sites + contrôles
m_c = smf.ols(f'uses_car ~ log_fee + lausanne + post + {controls}', data=est).fit(**cluster(est))
print(m_c.summary().tables[1])

c = m_c.params['log_fee']
print(f"\nEffet d'un doublement du prix : a × ln2 = {c * np.log(2) * 100:.1f} pp")
print(f"R² : (b) = {m_b.rsquared:.3f}  ->  (c) = {m_c.rsquared:.3f}")

Observations : 1992 | employés : 996
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3330      0.140      2.384      0.017       0.059       0.607
C(n_children)[T.1]     0.1656      0.035      4.735      0.000       0.097       0.234
C(n_children)[T.2]     0.1747      0.031      5.578      0.000       0.113       0.236
log_fee               -0.1883      0.023     -8.081      0.000      -0.234      -0.143
lausanne               0.0383      0.048      0.798      0.425      -0.056       0.132
post                   0.0300      0.012      2.479      0.013       0.006       0.054
transit_index         -0.1032      0.028     -3.697      0.000      -0.158      -0.048
distance_km            0.0281      0.003     10.465      0.000       0.023       0.033
income_10k             0.0272      0.004      6.091      0.000       0.018       0.036
age   

#### (d) Interactions : l'effet du prix varie-t-il selon le profil ?

$$\text{uses\_car}_{it} = \alpha + a \cdot \log(\text{fee}_{st}) + \underbrace{\log(\text{fee}_{st}) \times \big(\tilde T_i,\ \tilde Y_i,\ \text{Enfants}_i\big)}_{\text{interactions}} + \gamma \cdot \text{Lausanne}_s + \delta \cdot \text{Post}_t + \text{contrôles} + \varepsilon_{it}$$

**Codage des variables d'interaction**
- $\tilde T_i$ = `transit_c` : indice TP **centré** (indice − moyenne de l'échantillon, 0,457).
- $\tilde Y_i$ = `income_c` : revenu en 10 000 CHF **centré** (revenu − moyenne, 9,40, soit 94 000 CHF).
- Enfants : **catégories** 1 et 2+, avec **0 enfant comme groupe de référence**.

**Lecture**
- Grâce au centrage, *a* = effet du prix pour un employé à **l'accès TP moyen**, au **revenu moyen** et **sans enfant**.
- `log_fee:transit_c` : de combien l'effet du prix change pour +1 point d'indice TP. Négatif = les gens bien desservis réagissent **plus**.
- `log_fee:income_c` : de combien l'effet change pour +10 000 CHF de revenu.
- `log_fee:C(n_children)[T.2]` : écart d'effet entre les familles 2+ et les employés sans enfant.
- Les variables d'interaction restent aussi en **niveau** (`transit_c`, `income_c`, `C(n_children)`), pour que l'interaction ne mesure que la différence de **réaction** et pas la différence de niveau.

In [35]:
# Variables centrées (moyenne calculée sur l'échantillon d'estimation)
est = est.copy()
est['transit_c'] = est['transit_index'] - est['transit_index'].mean()
est['income_c'] = est['income_10k'] - est['income_10k'].mean()

# (d) (c) + interactions log(fee) × (TP, revenu, enfants)
formula_d = ('uses_car ~ log_fee + log_fee:transit_c + log_fee:income_c + log_fee:C(n_children)'
             ' + lausanne + post + transit_c + income_c + C(n_children) + distance_km + age + female + fte')
m_d = smf.ols(formula_d, data=est).fit(**cluster(est))
print(m_d.summary().tables[1])

# Effet d'un doublement du prix (en pp) pour quelques profils -> à comparer avec la Table B2
p = m_d.params
ln2 = np.log(2)
profils = {
    'Employé moyen, sans enfant':        p['log_fee'],
    'Accès TP +1 point (meilleur)':       p['log_fee'] + p['log_fee:transit_c'],
    'Accès TP -1 point (moins bon)':      p['log_fee'] - p['log_fee:transit_c'],
    'Revenu +30 000 CHF':                 p['log_fee'] + 3 * p['log_fee:income_c'],
    '1 enfant':                           p['log_fee'] + p['log_fee:C(n_children)[T.1]'],
    '2+ enfants':                         p['log_fee'] + p['log_fee:C(n_children)[T.2]'],
}
print("\nEffet d'un doublement du prix (2 -> 4 CHF), en pp :")
for k, v in profils.items():
    print(f"  {k:<32} {v * ln2 * 100:6.1f} pp")

                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                      0.5719      0.152      3.761      0.000       0.274       0.870
C(n_children)[T.1]             0.1835      0.069      2.652      0.008       0.048       0.319
C(n_children)[T.2]             0.0683      0.057      1.204      0.228      -0.043       0.179
log_fee                       -0.1903      0.031     -6.082      0.000      -0.252      -0.129
log_fee:C(n_children)[T.1]    -0.0168      0.058     -0.291      0.771      -0.130       0.096
log_fee:C(n_children)[T.2]     0.1102      0.046      2.376      0.017       0.019       0.201
log_fee:transit_c             -0.1236      0.030     -4.179      0.000      -0.182      -0.066
log_fee:income_c               0.0075      0.006      1.251      0.211      -0.004       0.019
lausanne                       0.0018      0.049  

In [ ]:
# --- Table B3 : les 4 spécifications sur le MÊME échantillon (1992 obs., 996 employés) ---
est_laus = est[est['site'] == 'Lausanne']
ctrl = 'transit_c + income_c + C(n_children) + distance_km + age + female + fte'   # mêmes contrôles (version centrée)

models = {
    '(a) Lausanne seule': smf.ols('uses_car ~ log_fee', data=est_laus).fit(**cluster(est_laus)),
    '(b) + Sion':         smf.ols('uses_car ~ log_fee + lausanne + post', data=est).fit(**cluster(est)),
    '(c) + contrôles':    smf.ols(f'uses_car ~ log_fee + lausanne + post + {ctrl}', data=est).fit(**cluster(est)),
    '(d) + interactions': smf.ols(formula_d, data=est).fit(**cluster(est)),
}

rows = {   # nom dans statsmodels -> libellé lisible (ordre d'affichage)
    'log_fee':                    'log(tarif)',
    'log_fee:transit_c':          'log(tarif) × indice TP (centré)',
    'log_fee:income_c':           'log(tarif) × revenu (centré, 10k CHF)',
    'log_fee:C(n_children)[T.1]': 'log(tarif) × 1 enfant',
    'log_fee:C(n_children)[T.2]': 'log(tarif) × 2+ enfants',
    'lausanne':                   'Lausanne (site)',
    'post':                       '2025 (année)',
    'transit_c':                  'Indice TP (centré)',
    'income_c':                   'Revenu (centré, 10k CHF)',
    'C(n_children)[T.1]':         '1 enfant',
    'C(n_children)[T.2]':         '2+ enfants',
    'distance_km':                'Distance (km)',
    'age':                        'Âge',
    'female':                     'Femme',
    'fte':                        "Taux d'activité",
    'Intercept':                  'Constante',
}

def stars(pv):
    return '***' if pv < 0.01 else '**' if pv < 0.05 else '*' if pv < 0.1 else ''

def cell(m, var):   # "coef*** (SE)" ou vide si la variable n'est pas dans le modèle
    if var not in m.params:
        return ''
    return f"{m.params[var]:.3f}{stars(m.pvalues[var])} ({m.bse[var]:.3f})"

table_B3 = pd.DataFrame({name: [cell(m, v) for v in rows] for name, m in models.items()},
                        index=list(rows.values()))

footer = pd.DataFrame({name: {
    "Effet d'un doublement du tarif (pp)": f"{m.params['log_fee'] * np.log(2) * 100:.1f}",
    'Observations':                        f"{int(m.nobs)}",
    'Employés (clusters)':                 f"{m.model.data.frame['employee_id'].nunique()}",
    'R²':                                  f"{m.rsquared:.3f}",
} for name, m in models.items()})

table_B3 = pd.concat([table_B3, footer])
table_B3

**Table B3 : équation (1), probabilité d'utiliser la voiture (modèle de probabilité linéaire)**

| | (a) Lausanne seule | (b) + Sion | (c) + contrôles | (d) + interactions |
|---|---:|---:|---:|---:|
| **log(tarif)** | **−0,145\*\*\*** (0,015) | **−0,188\*\*\*** (0,023) | **−0,188\*\*\*** (0,023) | **−0,190\*\*\*** (0,031) |
| log(tarif) × indice TP (centré) | | | | −0,124\*\*\* (0,030) |
| log(tarif) × revenu (centré, 10k CHF) | | | | 0,007 (0,006) |
| log(tarif) × 1 enfant | | | | −0,017 (0,058) |
| log(tarif) × 2+ enfants | | | | 0,110\*\* (0,046) |
| Lausanne (site) | | −0,044 (0,039) | 0,038 (0,048) | 0,002 (0,049) |
| 2025 (année) | | 0,030\*\* (0,012) | 0,030\*\* (0,012) | 0,030\*\* (0,012) |
| Indice TP (centré) | | | −0,103\*\*\* (0,028) | 0,018 (0,041) |
| Revenu (centré, 10k CHF) | | | 0,027\*\*\* (0,004) | 0,020\*\*\* (0,008) |
| 1 enfant | | | 0,166\*\*\* (0,035) | 0,183\*\*\* (0,069) |
| 2+ enfants | | | 0,175\*\*\* (0,031) | 0,068 (0,057) |
| Distance (km) | | | 0,028\*\*\* (0,003) | 0,028\*\*\* (0,003) |
| Âge | | | −0,001 (0,001) | −0,001 (0,001) |
| Femme | | | 0,043 (0,027) | 0,044 (0,027) |
| Taux d'activité | | | −0,184 (0,135) | −0,175 (0,134) |
| Constante | 0,642\*\*\* (0,023) | 0,716\*\*\* (0,036) | 0,541\*\*\* (0,151) | 0,572\*\*\* (0,152) |
| ***Effet d'un doublement du tarif (pp)*** | ***−10,1*** | ***−13,1*** | ***−13,1*** | ***−13,2*** |
| Observations | 1592 | 1992 | 1992 | 1992 |
| Employés (clusters) | 796 | 996 | 996 | 996 |
| R² | 0,010 | 0,016 | 0,236 | 0,240 |

*Variable dépendante : `uses_car` (1 = mode principal voiture). Les coefficients sont en unités de probabilité (0,01 = 1 point de pourcentage). Effet d'un doublement = coefficient de log(tarif) × ln 2. Erreurs standard groupées par employé entre parenthèses ; \* p < 0,1, \*\* p < 0,05, \*\*\* p < 0,01. Échantillon : 2024–2025, hors 4 employés sans revenu déclaré ; Lausanne seule en (a). Références : Sion (site), 2024 (année), 0 enfant. Indice TP et revenu centrés sur leur moyenne : en (d), le coefficient de log(tarif) est l'effet pour un employé à l'accès TP et au revenu moyens, sans enfant. En (d), les coefficients de niveau des variables interagies (indice TP, revenu, enfants) se lisent à log(tarif) = 0 et ne sont pas directement comparables à ceux de (c).*

# Part C — Intensive margin and decomposition

La marge intensive se calcule **uniquement sur les utilisateurs de la voiture** (`car_days_per_week > 0`). La marge totale porte sur **tous les employés**.

### 1. Compute the overall intensive elasticity from the change in mean driving days among car users. Then describe how this response differs by transit-accessibility group, income tercile, and number of children. State whether these are descriptive subgroup estimates or model-implied effects.

#### Required output: Table C1 with 2024 and 2025 mean car days among car users, intensive elasticity, and sample size for the full sample and each requested group.

**Élasticité intensive** = Δln(jours moyens par automobiliste) / Δln(prix) = ln(jours 2025 / jours 2024) / ln(4/2).
C'est la même forme en log que l'élasticité extensive de la question B2, ce qui permet d'additionner les deux en C.4. Comme en B2, on ne la calcule que pour **Lausanne**, seul site où le prix a changé. Les groupes sont les mêmes qu'en Partie B.

In [ ]:
# Échantillon de la marge intensive : utilisateurs de la voiture (car_days_per_week > 0)
users = raw[raw['car_days_per_week'] > 0].copy()
users['log_days'] = np.log(users['car_days_per_week'])

# Élasticité intensive = Δln(jours moyens) / Δln(prix) -> même définition que l'élasticité extensive (B2)
def int_elasticity(days_2024, days_2025, p0=2, p1=4):
    return np.log(days_2025 / days_2024) / np.log(p1 / p0)

def days_table(data, col):
    t = data.pivot_table(index=col, columns='year', values='car_days_per_week', aggfunc='mean', observed=True)
    t['delta_days'] = t[2025] - t[2024]
    t['elasticity'] = int_elasticity(t[2024], t[2025])
    n = data.pivot_table(index=col, columns='year', values='employee_id', aggfunc='nunique', observed=True)
    t['n_users_2024'], t['n_users_2025'] = n[2024], n[2025]
    t.index = pd.MultiIndex.from_product([[col], t.index.astype(str)], names=['variable', 'group'])
    return t

users_laus = users[users['site'] == 'Lausanne']
table_C1 = pd.concat([days_table(users_laus.assign(all='all'), 'all')] +
                     [days_table(users_laus, c) for c in ['transit_group', 'income_tercile', 'children_group']])

# Pour référence : Sion (prix inchangé -> pas d'élasticité)
print("Sion, jours moyens des automobilistes :",
      users[users.site == 'Sion'].groupby('year')['car_days_per_week'].mean().round(2).to_dict())
table_C1.round(2)

**Table C1 : jours de voiture par semaine parmi les automobilistes, site de Lausanne (tarif 2 → 4 CHF)**

| Variable | Groupe | Jours 2024 | Jours 2025 | Δ jours | Élasticité intensive | n automobilistes 2024 | n automobilistes 2025 |
|---|---|---:|---:|---:|---:|---:|---:|
| **Total** | Lausanne | 3,60 | 3,45 | −0,15 | **−0,06** | 431 | 351 |
| **Accessibilité TP** | 1_low | 3,86 | 3,73 | −0,14 | −0,05 | 213 | 196 |
| | 2_medium | 3,48 | 3,31 | −0,18 | −0,07 | 120 | 88 |
| | 3_high | 3,19 | 2,84 | −0,35 | −0,17 | 98 | 67 |
| **Tercile de revenu** | 1_low (< 77 500 CHF) | 3,44 | 3,28 | −0,16 | −0,07 | 125 | 92 |
| | 2_medium (77 500 – 103 200 CHF) | 3,64 | 3,52 | −0,13 | −0,05 | 135 | 113 |
| | 3_high (> 103 200 CHF) | 3,69 | 3,51 | −0,18 | −0,07 | 171 | 146 |
| **Nombre d'enfants** | 0 | 3,26 | 2,93 | −0,34 | −0,16 | 190 | 143 |
| | 1 | 3,54 | 3,48 | −0,07 | −0,03 | 100 | 77 |
| | 2+ | 4,11 | 4,01 | −0,09 | −0,03 | 141 | 131 |

*Élasticité intensive = ln(jours 2025 / jours 2024) / ln(4/2). Échantillon : employés de Lausanne avec `car_days_per_week > 0` l'année considérée (la valeur impossible de 7,4 jours est plafonnée à 5). Pour référence, à Sion (prix inchangé), les jours moyens passent de 3,55 à 3,65.*

**Lecture**
- **L'effet global est faible** : les automobilistes qui restent conduisent 0,15 jour de moins par semaine, soit une élasticité de −0,06, cinq fois plus faible que l'élasticité extensive (−0,30).
- **On retrouve la même hiérarchie qu'en B2** : ceux qui ont une alternative en TP (3_high : −0,17) ou sans enfant (−0,16) réduisent le plus leurs jours. Les familles (1 enfant et 2+ : −0,03) et les mal desservis (1_low : −0,05) ne changent presque rien.
- **Revenu** : pas de différence nette entre terciles (−0,05 à −0,07).
- **Ce sont des estimations descriptives par sous-groupe**, pas des effets issus d'un modèle : de simples moyennes avant/après, sans contrôles et sans Sion. Les groupes se recoupent, et la composition des automobilistes change entre 2024 et 2025 (voir C.3).

### 2. Estimate equation (2) sequentially: Lausanne only; then with Sion; then with controls; and finally with controls and interactions.

#### Required output: Table C2, a compact regression table with the four specifications (policy coefficient in native units and elasticity, uncertainty, observations, R², sample restrictions).

$$\log(\text{car\_days\_week}_{it}) = \alpha + b \cdot \log(\text{fee}_{st}) + \text{contrôles} + \log(\text{fee}_{st}) \times \text{interactions} + \varepsilon_{it} \qquad (\text{automobilistes seulement})$$

- **Même logique qu'en Partie B** : (a) Lausanne avant/après, (b) + Sion avec `lausanne` et `post`, (c) + les mêmes contrôles, (d) + les mêmes interactions centrées (référence : 0 enfant).
- **Unités** : la variable dépendante et le tarif sont tous deux en log, donc *b* est **directement une élasticité** : +1 % de tarif entraîne *b* % de jours de voiture en plus (ou en moins).
- **Échantillon** : uniquement les observations avec `car_days_per_week > 0`, hors employés sans revenu. Le panel n'est **pas équilibré** : un employé qui quitte la voiture en 2025 n'apparaît qu'en 2024. Erreurs standard groupées par employé.

In [ ]:
# Échantillon : automobilistes avec revenu connu, variables centrées comme en Partie B
users_est = users.dropna(subset=['income_10k']).copy()
users_est['transit_c'] = users_est['transit_index'] - est['transit_index'].mean()   # même centrage qu'en B
users_est['income_c'] = users_est['income_10k'] - est['income_10k'].mean()
users_est_laus = users_est[users_est['site'] == 'Lausanne']

# Les 4 spécifications de l'équation (2) : mêmes formules qu'en B, avec log_days à la place de uses_car
formula_C = {
    '(a) Lausanne seule': 'log_days ~ log_fee',
    '(b) + Sion':         'log_days ~ log_fee + lausanne + post',
    '(c) + contrôles':    f'log_days ~ log_fee + lausanne + post + {ctrl}',
    '(d) + interactions': formula_d.replace('uses_car', 'log_days'),
}
models_C = {}
for name, f in formula_C.items():
    data = users_est_laus if name.startswith('(a)') else users_est
    models_C[name] = smf.ols(f, data=data).fit(**cluster(data))

# Table C2 compacte : variables de politique (+ site / année), contrôles signalés par "oui / non"
rows_C = {k: v for k, v in rows.items() if k.startswith('log_fee') or k in ('lausanne', 'post')}
table_C2 = pd.DataFrame({name: [cell(m, v) for v in rows_C] for name, m in models_C.items()},
                        index=list(rows_C.values()))
footer_C = pd.DataFrame({name: {
    'Contrôles':           'non' if name[:3] in ('(a)', '(b)') else 'oui',
    'Observations':        f"{int(m.nobs)}",
    'Employés (clusters)': f"{m.model.data.frame['employee_id'].nunique()}",
    'R²':                  f"{m.rsquared:.3f}",
} for name, m in models_C.items()})
table_C2 = pd.concat([table_C2, footer_C])
table_C2

**Table C2 : équation (2), log(jours de voiture par semaine), automobilistes seulement**

| | (a) Lausanne seule | (b) + Sion | (c) + contrôles | (d) + interactions |
|---|---:|---:|---:|---:|
| **log(tarif) = élasticité intensive** | **−0,077\*\*\*** (0,015) | **−0,114\*\*\*** (0,022) | **−0,148\*\*\*** (0,021) | **−0,236\*\*\*** (0,027) |
| log(tarif) × indice TP (centré) | | | | −0,069\*\*\* (0,020) |
| log(tarif) × revenu (centré, 10k CHF) | | | | 0,002 (0,004) |
| log(tarif) × 1 enfant | | | | 0,157\*\*\* (0,035) |
| log(tarif) × 2+ enfants | | | | 0,138\*\*\* (0,028) |
| Lausanne (site) | | 0,010 (0,020) | 0,050\*\* (0,020) | 0,038\* (0,020) |
| 2025 (année) | | 0,026\*\* (0,012) | 0,022\* (0,012) | 0,022\* (0,012) |
| Contrôles | non | non | oui | oui |
| Observations | 782 | 1022 | 1022 | 1022 |
| Employés (clusters) | 431 | 554 | 554 | 554 |
| R² | 0,012 | 0,013 | 0,499 | 0,513 |

*Variable dépendante : log(`car_days_per_week`). Échantillon : observations avec `car_days_per_week` > 0, 2024–2025, hors employés sans revenu ; Lausanne seule en (a). Le coefficient de log(tarif) est directement une élasticité. Contrôles : indice TP, revenu (centrés), enfants (référence : 0), distance, âge, femme, taux d'activité. En (d), le coefficient principal est l'élasticité d'un automobiliste à l'accès TP et au revenu moyens, sans enfant. Erreurs standard groupées par employé ; \* p < 0,1, \*\* p < 0,05, \*\*\* p < 0,01.*

**Comparaison des spécifications**
- **(a) → (b)** : comme en Partie B, Sion corrige la tendance commune. Les automobilistes de Sion conduisent un peu **plus** en 2025 (+2,6 %), donc l'effet de Lausanne passe de −0,08 à −0,11.
- **(b) → (c)** : contrairement à la Partie B, **les contrôles changent le résultat** (−0,11 → −0,15). Ici, l'échantillon n'est plus le même entre 2024 et 2025, puisque ceux qui ont quitté la voiture sortent de l'échantillon. Les contrôles corrigent en partie ce changement de composition (voir C.3).
- **(d)** : même schéma qu'en Partie B. Les automobilistes bien desservis en TP réduisent plus leurs jours. Les parents (1 et 2+ enfants) réduisent beaucoup moins : −0,236 + 0,138 ≈ −0,10 pour les 2+. Le revenu ne joue pas.
- **Ordre de grandeur** : même dans la spécification la plus forte, la marge intensive (−0,15) reste **inférieure** à la marge extensive (−0,30 en B2).

### 3. Compare the estimates across specifications. Discuss selection into the car-user sample: is the average car user in 2025 the same kind of person as the average car user in 2024?

Trois vérifications :
1. comparer le **profil moyen** des automobilistes de Lausanne en 2024 et en 2025 ;
2. comparer, en 2024, ceux qui **restent** en voiture (*stayers*) et ceux qui la **quittent** en 2025 (*quitters*) ;
3. recalculer l'élasticité intensive sur les seuls *stayers*, c'est-à-dire les mêmes personnes les deux années, donc sans effet de composition.

In [ ]:
# 1) Les automobilistes de 2025 ressemblent-ils à ceux de 2024 ? (Lausanne)
selection = (users_laus.groupby('year')[['car_days_per_week', 'distance_km', 'transit_index',
                                          'income_10k', 'n_children', 'age']].mean().T)
selection['diff'] = selection[2025] - selection[2024]
print("Profil moyen des automobilistes de Lausanne :")
print(selection.round(3))

# 2) Qui a quitté la voiture ? Comparaison 'stayers' / 'quitters' en 2024
status = raw[raw.site == 'Lausanne'].pivot(index='employee_id', columns='year', values='uses_car')
stayers = status[(status[2024] == 1) & (status[2025] == 1)].index
quitters = status[(status[2024] == 1) & (status[2025] == 0)].index
lau24 = raw[(raw.site == 'Lausanne') & (raw.year == 2024)].set_index('employee_id')
cols = ['car_days_per_week', 'distance_km', 'transit_index', 'income_10k']
comp = pd.DataFrame({
    'stayers (voiture 2024 et 2025)':    lau24.loc[stayers, cols].mean(),
    'quitters (voiture 2024 seulement)': lau24.loc[quitters, cols].mean(),
}).round(2)
comp.loc['n employés'] = [len(stayers), len(quitters)]
print("\nEn 2024 :")
print(comp)

# 3) Élasticité intensive sur les seuls 'stayers' (mêmes personnes les deux années -> pas de sélection)
st = raw[(raw.site == 'Lausanne') & raw.employee_id.isin(stayers)].groupby('year')['car_days_per_week'].mean()
print(f"\nStayers : {st[2024]:.2f} -> {st[2025]:.2f} jours ; élasticité intensive = {int_elasticity(st[2024], st[2025]):.3f}")
print(f"Tous les automobilistes (Table C1) : élasticité intensive = {table_C1.loc[('all', 'all'), 'elasticity']:.3f}")

**Résultats : non, l'automobiliste moyen de 2025 n'est pas le même que celui de 2024.**

| Lausanne, en 2024 | Stayers (voiture 2024 et 2025) | Quitters (voiture 2024 seulement) |
|---|---:|---:|
| n employés | 351 | 80 |
| Jours de voiture / semaine | 3,73 | **3,07** |
| Distance (km) | 11,1 | **7,0** |
| Indice TP | 0,41 | **0,87** |
| Revenu (10k CHF) | 10,2 | 9,2 |

- **Ceux qui ont quitté la voiture** (80 employés) habitaient plus près, avaient un meilleur accès TP et conduisaient déjà **moins de jours** (3,07 contre 3,73).
- Ils sortent de l'échantillon des automobilistes en 2025. Les automobilistes restants ont donc, en moyenne, des trajets plus longs (11,1 km contre 10,4), un moins bon accès TP et **plus de jours de voiture au départ**. C'est un **biais de sélection** : en 2025, la moyenne porte sur un groupe de « gros conducteurs ».
- **Conséquence** : la simple comparaison des moyennes (Table C1, −0,06) **sous-estime** la réponse intensive. Sur les seuls *stayers* (mêmes personnes), les jours passent de 3,73 à 3,45, soit une **élasticité de −0,11**, presque le double. C'est aussi pour ça que l'ajout des contrôles en Table C2 (b → c) rend l'élasticité plus forte (−0,11 → −0,15) : les contrôles tiennent compte en partie du fait que les automobilistes de 2025 sont plus éloignés et moins bien desservis.
- **Pour le rapport** : l'élasticité intensive « brute » est une **borne basse**. Une valeur entre −0,11 (stayers) et −0,15 (avec contrôles) est plus crédible.

### 4. Verify the decomposition using mutually consistent definitions and samples: ε_total = ε_extensive + ε_intensive. State which margin accounts for most of the total change.

#### Required output: Table C3 reporting the extensive, intensive, and total elasticities, together with the underlying means used in the calculation.

**Pourquoi la décomposition marche** : pour tout le groupe d'employés,

$$\underbrace{\text{jours voiture par employé}}_{\text{total}} = \underbrace{\text{part d'automobilistes}}_{\text{extensive}} \times \underbrace{\text{jours par automobiliste}}_{\text{intensive}}$$

En prenant le log, un produit devient une somme : ln(total) = ln(part) + ln(jours). En divisant la variation par Δln(prix) = ln 2, on obtient **exactement** ε_total = ε_extensive + ε_intensive.

**Échantillon cohérent** : les **800 employés de Lausanne** les deux années (les non-automobilistes comptent pour 0 jour dans le total). On utilise les mêmes définitions en log qu'en B2 et C1.

In [ ]:
# Décomposition sur le MÊME échantillon : les 800 employés de Lausanne
lau = raw[raw['site'] == 'Lausanne']
m = lau.groupby('year').agg(
    car_share=('uses_car', 'mean'),                                              # marge extensive
    days_per_user=('car_days_per_week', lambda s: s[s > 0].mean()),              # marge intensive
    days_per_employee=('car_days_per_week', 'mean'))                             # total (non-automobilistes = 0)

# Identité : jours par employé = part voiture × jours par automobiliste
assert np.allclose(m['days_per_employee'], m['car_share'] * m['days_per_user'])

eps = {col: np.log(m.loc[2025, col] / m.loc[2024, col]) / np.log(4 / 2) for col in m.columns}
table_C3 = pd.DataFrame({
    'Mesure':       ['Part voiture (extensive)', 'Jours par automobiliste (intensive)', 'Jours voiture par employé (totale)'],
    'Moyenne 2024': [m.loc[2024, c] for c in m.columns],
    'Moyenne 2025': [m.loc[2025, c] for c in m.columns],
    'Élasticité':   [eps[c] for c in m.columns],
}).set_index('Mesure')
table_C3['Part du total (%)'] = table_C3['Élasticité'] / eps['days_per_employee'] * 100

print(f"Vérification : ε_ext + ε_int = {eps['car_share'] + eps['days_per_user']:.4f}  |  ε_total = {eps['days_per_employee']:.4f}")
table_C3.round(3)

**Table C3 : décomposition de l'élasticité totale, site de Lausanne (800 employés, tarif 2 → 4 CHF)**

| Marge | Mesure | Moyenne 2024 | Moyenne 2025 | Élasticité | Part du total |
|---|---|---:|---:|---:|---:|
| Extensive | Part d'automobilistes | 0,539 | 0,439 | **−0,296** | **83 %** |
| Intensive | Jours / semaine par automobiliste | 3,604 | 3,453 | **−0,062** | **17 %** |
| **Totale** | **Jours voiture / semaine par employé** | **1,942** | **1,515** | **−0,358** | 100 % |

*Élasticité = ln(moyenne 2025 / moyenne 2024) / ln(4/2). Vérification : −0,296 + (−0,062) = −0,358 = ε_total (identité exacte, car jours par employé = part × jours par automobiliste, pour chaque année). Échantillon : les 800 employés de Lausanne ; les non-automobilistes comptent 0 jour dans le total.*

**Conclusion**
- La décomposition est **vérifiée exactement** : ε_total (−0,358) = ε_extensive (−0,296) + ε_intensive (−0,062).
- **La marge extensive explique l'essentiel (83 %) de la baisse** : le doublement du tarif agit surtout en faisant **abandonner la voiture** à certains employés, beaucoup moins en faisant conduire **moins souvent** ceux qui la gardent.
- **Nuance (voir C.3)** : cette décomposition descriptive utilise la moyenne brute des automobilistes, qui sous-estime la marge intensive à cause de la sélection. Avec l'élasticité des *stayers* (−0,11), la marge intensive pèserait environ un quart du total. La marge extensive resterait de loin la principale.
- Ces élasticités sont des comparaisons **avant/après à Lausanne**, sans Sion. Elles sont donc cohérentes avec les Tables B2 et C1, mais ne retirent pas la tendance commune estimée en B3 et C2.